<a href="https://colab.research.google.com/github/Noddyonthemoon/MSRIT-Results-scraper-Fixed/blob/master/recipe_react_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2: RAG-based ReAct recipe agent

This notebook is an introduction to a RAG-based ReAct agent. The agent looks through a recipe book, reasons about the best course of action, calls tools, and generates a response grounded in what it found.

**ReAct loop used here:** the model writes a short *thought*, requests a tool call (*act*), we run the tool and hand back the *observation*, and the model repeats until it can give a final answer.

## Setup - Gemini API key

1. Go to [aistudio.google.com](https://aistudio.google.com) and sign in with your personal Gmail account.
2. Accept the terms of service if prompted.
3. Click **Get API key** (the key icon) and create a FREE Google Cloud project / "Create API key in new project".
4. Copy the key (it starts with `AIza...`).
5. In Colab, open the **Secrets** (key icon) tab in the left sidebar, add a secret named `GOOGLE_API_KEY`, paste the key, and turn on notebook access.

The free tier allows roughly 10 requests per minute and 250 per day, which is enough for this notebook. If you see slow-downs when running several questions back to back, that is normal and the code retries automatically.

## References

- Google agent loop / thought signatures: https://ai.google.dev/gemini-api/docs/thinking#signatures
- Text generation and function calling: https://ai.google.dev/gemini-api/docs/text-generation

In [1]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')
# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to GitHub. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

Paste your Gemini API key and press Enter: ··········
API key set.


In [2]:
# Quiet keeps the output clean. google-genai is the current SDK
# (the older google-generativeai package is deprecated and is not used below).
!pip install -U google-genai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.
Done.


## Create a database of recipes

Each recipe is stored as one text document. Lines are joined with newlines so ingredients and steps stay readable when the agent retrieves them.

In [3]:
# @title Create a database of recipes.
RECIPES = {
    "bombe_alaska": "\n".join([
        "INGREDIENTS",
        "1 pint vanilla ice cream, softened",
        "1 pint raspberry sorbet, softened",
        "1 pint chocolate ice cream, softened",
        "1 prepared 9-inch sponge cake or genoise layer",
        "4 large egg whites",
        "1/2 cup granulated sugar",
        "Pinch of cream of tartar",
        "STEPS",
        "Line a 2-quart bowl with plastic wrap, leaving an overhang. Spoon vanilla ice cream into the bowl, spreading evenly. Freeze for 30 minutes.",
        "Repeat with raspberry sorbet, then chocolate ice cream, freezing each layer for 30 minutes.",
        "Place the cake layer on top of the chocolate ice cream, pressing gently. Cover with plastic wrap and freeze overnight.",
        "Preheat oven to 500°F (260°C). In a large bowl, whisk egg whites until foamy. Gradually add sugar and cream of tartar, whisking until stiff glossy peaks form (meringue).",
        "Unmold the ice cream dome onto an oven-safe serving plate. Remove plastic wrap. Spread meringue all over the ice cream dome, ensuring it's completely covered and sealed to the plate.",
        "Bake for 3-5 minutes or until meringue is golden brown. Serve immediately.",
    ]),
    "fried_rice": "\n".join([
        "INGREDIENTS",
        "3 cups cooked day-old white rice",
        "2 tbsp vegetable oil",
        "1/2 cup diced carrots",
        "1/2 cup frozen peas",
        "1/2 cup chopped onion",
        "2 cloves garlic, minced",
        "2 large eggs, beaten",
        "2 tbsp soy sauce",
        "1 tbsp oyster sauce (optional)",
        "1 tsp sesame oil",
        "Salt and pepper to taste",
        "Green onions, chopped, for garnish",
        "STEPS",
        "Heat vegetable oil in a large skillet or wok over medium-high heat.",
        "Add carrots, peas and onion. Cook, stirring frequently, for 3-5 minutes until vegetables are tender-crisp.",
        "Add minced garlic and cook for another minute until fragrant.",
        "Push vegetables to one side of the skillet. Pour beaten eggs into the empty side and scramble until cooked through. Break into small pieces.",
        "Add cooked rice to the skillet. Pour soy sauce, oyster sauce (if using) and sesame oil over the rice. Stir everything together, breaking up any clumps of rice.",
        "Cook for 5-7 minutes, stirring occasionally, until the rice is heated through and slightly browned.",
        "Season with salt and pepper to taste. Garnish with chopped green onions before serving.",
    ]),
    "butter_chicken": "\n".join([
        "INGREDIENTS",
        "1.5 lbs boneless skinless chicken thighs, cut into 1-inch pieces",
        "1/2 cup plain yogurt",
        "1 tbsp lemon juice",
        "2 tsp ground cumin",
        "2 tsp garam masala",
        "1 tsp turmeric powder",
        "1 tsp chili powder (or paprika for milder taste)",
        "1/2 tsp salt",
        "2 tbsp butter",
        "1 large onion, finely chopped",
        "2 cloves garlic, minced",
        "1 tbsp grated fresh ginger",
        "1 (14.5 ounce) can crushed tomatoes",
        "1/2 cup heavy cream",
        "1/4 cup chopped fresh cilantro, for garnish",
        "STEPS",
        "In a large bowl, combine chicken pieces with yogurt, lemon juice, cumin, garam masala, turmeric, chili powder and salt. Mix well to coat the chicken. Marinate for at least 30 minutes, or preferably 2-4 hours in the refrigerator.",
        "Melt butter in a large skillet or Dutch oven over medium heat. Add chopped onion and cook until softened and translucent, about 5-7 minutes.",
        "Add minced garlic and grated ginger; cook for 1 minute until fragrant.",
        "Add the marinated chicken to the skillet. Cook, stirring occasionally, until browned on all sides, about 8-10 minutes.",
        "Stir in the crushed tomatoes. Bring to a simmer, then reduce heat to low, cover and cook for 15-20 minutes or until chicken is cooked through.",
        "Stir in the heavy cream. Cook for another 5 minutes, uncovered, until the sauce thickens slightly. Adjust seasoning if needed.",
        "Serve hot over rice or with naan bread, garnished with fresh cilantro.",
    ]),
    "alfredo_pasta": "\n".join([
        "INGREDIENTS",
        "1 lb fettuccine pasta",
        "1/2 cup unsalted butter",
        "2 cups heavy cream",
        "1.5 cups freshly grated Parmesan cheese",
        "1/4 tsp ground nutmeg (optional)",
        "Salt and freshly ground black pepper to taste",
        "Fresh parsley, chopped, for garnish",
        "STEPS",
        "Cook fettuccine according to package directions until al dente. Reserve 1 cup of pasta water before draining.",
        "While pasta cooks, melt butter in a large skillet or saucepan over medium-low heat.",
        "Pour in heavy cream and bring to a gentle simmer, stirring occasionally. Do not boil.",
        "Reduce heat to low. Gradually whisk in the Parmesan cheese until melted and the sauce is smooth. If the sauce is too thick, add a little reserved pasta water until desired consistency is reached.",
        "Season with nutmeg (if using), salt and freshly ground black pepper. Taste and adjust seasoning.",
        "Add the drained fettuccine to the sauce and toss to coat evenly.",
        "Serve immediately, garnished with chopped fresh parsley.",
    ]),
}
print(f"Loaded {len(RECIPES)} recipes: {', '.join(RECIPES)}")

Loaded 4 recipes: bombe_alaska, fried_rice, butter_chicken, alfredo_pasta


## Define the tools

**Tool 1 - `search_recipes` (the search tool).** Matches whole words (not substrings), ignores filler words like "the" and "how", and treats simple plurals as the same word. The previous version matched any substring, so a word like "the" matched every recipe.

**Tool 2 - `recommend_substitution`.** A second action the agent can take, so it has more than one thing to reason about.

In [4]:
# @title Define the search tool
import re

STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "in", "on", "for", "with", "is", "are",
    "do", "does", "how", "what", "which", "can", "i", "we", "our", "my", "me", "you",
    "use", "uses", "be", "it", "this", "that", "from", "at", "by", "as", "long", "must",
    "cook", "make",
}


def _tokens(text: str, drop_stopwords: bool = False) -> set:
    """Lowercase word set; strips punctuation and folds simple plurals (eggs -> egg)."""
    # Words of 2+ characters only, so apostrophe leftovers like the "s" in "it's" never match.
    words = re.findall(r"[a-z0-9]{2,}", text.lower())
    if drop_stopwords:
        words = [w for w in words if w not in STOPWORDS]
    return {w[:-1] if len(w) > 3 and w.endswith("s") else w for w in words}


def search_recipes(query: str) -> str:
    """Search the cookbook collection for recipes relevant to the query."""
    query_terms = _tokens(query, drop_stopwords=True)
    results = []

    for doc_name, content in RECIPES.items():
        doc_terms = _tokens(content) | _tokens(doc_name.replace("_", " "))
        score = len(query_terms & doc_terms)
        if score > 0:
            results.append((score, doc_name, content))

    results.sort(key=lambda r: r[0], reverse=True)

    if not results:
        return "No relevant recipes found for that query."

    # Return the top two matches
    output = [f"[Source: {name}]\n{content}" for _, name, content in results[:2]]
    return "\n\n".join(output)


# Quick test
print(search_recipes("fresh parsley"))

[Source: alfredo_pasta]
INGREDIENTS
1 lb fettuccine pasta
1/2 cup unsalted butter
2 cups heavy cream
1.5 cups freshly grated Parmesan cheese
1/4 tsp ground nutmeg (optional)
Salt and freshly ground black pepper to taste
Fresh parsley, chopped, for garnish
STEPS
Cook fettuccine according to package directions until al dente. Reserve 1 cup of pasta water before draining.
While pasta cooks, melt butter in a large skillet or saucepan over medium-low heat.
Pour in heavy cream and bring to a gentle simmer, stirring occasionally. Do not boil.
Reduce heat to low. Gradually whisk in the Parmesan cheese until melted and the sauce is smooth. If the sauce is too thick, add a little reserved pasta water until desired consistency is reached.
Season with nutmeg (if using), salt and freshly ground black pepper. Taste and adjust seasoning.
Add the drained fettuccine to the sauce and toss to coat evenly.
Serve immediately, garnished with chopped fresh parsley.

[Source: butter_chicken]
INGREDIENTS
1.5 l

In [5]:
# @title Define the substitution tool
SUBSTITUTIONS = {
    "butter": ["margarine", "coconut oil", "olive oil", "applesauce (for baking)"],
    "heavy cream": ["half-and-half with butter", "whole milk with butter", "coconut cream (dairy-free)", "evaporated milk"],
    "parmesan cheese": ["pecorino romano", "grana padano", "nutritional yeast (vegan option)"],
    "chicken thighs": ["chicken breasts", "firm tofu (vegetarian option)", "turkey thighs", "pork shoulder"],
    "soy sauce": ["tamari (gluten-free)", "coconut aminos", "liquid aminos"],
    "egg": ["applesauce", "mashed banana", "chia seed egg", "flaxseed egg", "commercial egg replacer"],
    "yogurt": ["sour cream", "greek yogurt", "coconut yogurt (dairy-free)", "buttermilk"],
}


def recommend_substitution(ingredient: str) -> str:
    """Recommend common culinary substitutions for a given ingredient."""
    ingredient_lower = ingredient.strip().lower()

    # Direct match check
    if ingredient_lower in SUBSTITUTIONS:
        subs = ", ".join(SUBSTITUTIONS[ingredient_lower])
        return f"Substitutions for '{ingredient}': {subs}"

    # Fuzzy match check for substrings
    matches = []
    for key, values in SUBSTITUTIONS.items():
        if key in ingredient_lower or ingredient_lower in key:
            subs = ", ".join(values)
            matches.append(f"Substitutions for '{key}': {subs}")

    if matches:
        return "\n".join(matches)

    return f"No direct substitution found in our database for '{ingredient}'."


# Quick test
print(recommend_substitution("heavy cream"))

Substitutions for 'heavy cream': half-and-half with butter, whole milk with butter, coconut cream (dairy-free), evaporated milk


## Configure Gemini with the tools

The earlier version of this notebook built a model with the deprecated `google.generativeai` package, but the agent loop actually called the newer `google.genai` client, so the **system instruction was never applied**. Here everything goes through one client, and the system instruction is passed in the request config.

In [6]:
# @title Configure the Gemini client, tools and system instruction
import time
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.5-flash-lite"  # swap for any model listed in AI Studio if you get a 404

client = genai.Client()  # reads GOOGLE_API_KEY from the environment

SYSTEM_INSTRUCTION = """
You are a chef training at a Michelin star restaurant.
You work in a ReAct style: before each tool call, write ONE short sentence of reasoning
about what you need and why.
Use the search_recipes tool to look up temperatures, ingredients, cook times and steps.
Use the recommend_substitution tool when an ingredient needs to be swapped (dietary needs, missing items).
Only answer based on what you find with the tools. Do not use outside knowledge.
If the tools do not return the answer, say so clearly.
"""

recipe_tools = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_recipes",
        description="Search the restaurant's cookbook and return the most relevant recipes (ingredients and steps).",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string", "description": "Keywords such as a dish name or ingredient."}},
            "required": ["query"],
        },
    ),
    types.FunctionDeclaration(
        name="recommend_substitution",
        description="Suggest substitutes for a single ingredient (e.g. 'heavy cream').",
        parameters={
            "type": "object",
            "properties": {"ingredient": {"type": "string", "description": "The ingredient to replace."}},
            "required": ["ingredient"],
        },
    ),
])

# Maps the tool name the model asks for to the Python function that runs it.
TOOL_REGISTRY = {
    "search_recipes": search_recipes,
    "recommend_substitution": recommend_substitution,
}

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    tools=[recipe_tools],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)
print("Client ready. Model:", MODEL_NAME)

Client ready. Model: gemini-3.5-flash-lite


## The ReAct agent loop

Each step: the model reasons (*thought*) and may request tool calls (*action*); we run them and return the results (*observation*). The loop ends when the model answers without requesting a tool, or after `max_steps`.

In [7]:
# @title ReAct agent loop
def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME, contents=contents, config=config
            )
        except Exception as e:
            rate_limited = "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)
            if rate_limited and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_tool(call) -> str:
    """Run one requested tool call and return its result as text."""
    fn = TOOL_REGISTRY.get(call.name)
    if fn is None:
        return f"Error: unknown tool '{call.name}'."
    try:
        return fn(**dict(call.args))
    except Exception as e:
        return f"Error while running {call.name}: {e}"


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content
        parts = model_content.parts or []

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # THOUGHT: any reasoning text the model wrote alongside its tool calls
        thought = " ".join(p.text.strip() for p in parts if p.text)
        if thought:
            print(f"Step {step} thought: {thought}")

        # ACT + OBSERVE: run every requested tool, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            print(f"Step {step} action: {call.name}({dict(call.args)})")
            result = run_tool(call)
            print(f"  Observation: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(name=call.name, response={"result": result})
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"

## Try it out

In [8]:
_ = run_agent("What can I cook with vanilla and caramel, suggest a substitution for caramel")

Question: What can I cook with vanilla and caramel, suggest a substitution for caramel
--------------------------------------------------
Step 1 action: search_recipes({'query': 'vanilla caramel'})
  Observation: [Source: bombe_alaska]
INGREDIENTS
1 pint vanilla ice cream, softened
1 pint raspberry sorbet, softe...
Step 2 action: search_recipes({'query': 'caramel'})
  Observation: No relevant recipes found for that query....
Step 3 action: search_recipes({'query': 'vanilla'})
  Observation: [Source: bombe_alaska]
INGREDIENTS
1 pint vanilla ice cream, softened
1 pint raspberry sorbet, softe...
Step 4 action: recommend_substitution({'ingredient': 'caramel'})
  Observation: No direct substitution found in our database for 'caramel'....

Final answer:
None


In [9]:
questions = [
    "What is the best chicken dish on our menu?",
    "Does our restaurant's pasta use ravioli or linguine?",
    "How long must the rice be cooked for",
    # Needs both tools: find the recipe, then look up substitutions.
    "I'm vegan. What would I need to swap in the alfredo pasta?",
    # This question is not answered by our database.
    "How many customers eat our food everyday?",
    # This question is part of the general knowledge of the model, but the agent should refuse.
    "What is the most famous food item in Japan?",
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


Question: What is the best chicken dish on our menu?
--------------------------------------------------
Step 1 thought: I will search our cookbook for any chicken dishes to see what is on the menu.
Step 1 action: search_recipes({'query': 'chicken'})
  Observation: [Source: butter_chicken]
INGREDIENTS
1.5 lbs boneless skinless chicken thighs, cut into 1-inch piece...

Final answer:
Our cookbook features Butter Chicken, made with marinated boneless skinless chicken thighs, yogurt, lemon juice, spices, butter, onions, garlic, ginger, crushed tomatoes, and heavy cream.

Question: Does our restaurant's pasta use ravioli or linguine?
--------------------------------------------------
Step 1 action: search_recipes({'query': 'pasta'})
  Observation: [Source: alfredo_pasta]
INGREDIENTS
1 lb fettuccine pasta
1/2 cup unsalted butter
2 cups heavy cream...

Final answer:
Based on our cookbook, the restaurant's pasta recipe (Alfredo pasta) uses fettuccine pasta, neither ravioli nor linguine.

Quest